### i need to get date range for API parameters, so for that i am going to make a function that will deal with it


In [8]:
from datetime import datetime as dt
from datetime import timedelta as td

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

end_date = dt.now().strftime("%Y-%m-%d")
end_date

'2026-07-27'

In [9]:
start_date = dt.now() - td(days=7)
start_date = start_date.strftime("%y-%m-%d")
start_date

'26-07-20'

### this reminds me of the operator overloading i learned, notice we are subtracting class from class object.

### its working because in module we can define `__sub__` and control its behavior which allows it to handle such things


In [10]:
def parameter_builder(file_path):
    df = pd.read_csv(file_path)

    # calcualting date based on current date
    end_date = dt.now().strftime("%Y-%m-%d")
    diff = dt.now() - td(days=7)
    start_date = diff.strftime("%Y-%m-%d")

    for _, row in df.iterrows():
        params = {
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
            "timezone": "auto",
            "start_date": start_date,
            "end_date": end_date,
        }

        yield row["site_code"], params

In [11]:
# lets see if it works as intended
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    print(i, param)

0 ('HBRPO', {'latitude': 33.93991, 'longitude': 73.27844, 'hourly': ['temperature_2m', 'relative_humidity_2m', 'shortwave_radiation'], 'timezone': 'auto', 'start_date': '2026-07-20', 'end_date': '2026-07-27'})


### ok now i am gonna need a function that will fetch the data


In [16]:
from datetime import datetime as dt
from datetime import timedelta as td
from retry_requests import retry
from sqlalchemy import engine

import openmeteo_requests
import pandas as pd
import requests_cache

# retries and backoff factors can handle errors
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://api.open-meteo.com/v1/forecast"


def fetch_weather_data(site_code, params):
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_global_tilted_irradiance_instant = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )
    }

    hourly_data["site_code"] = site_code
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["global_tilted_irradiance_instant"] = (
        hourly_global_tilted_irradiance_instant
    )

    hourly_dataframe = pd.DataFrame(data=hourly_data)
    print("\nHourly data\n", hourly_dataframe)

### Testing the output we get


In [19]:
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    fetch_weather_data(site_code=param[0], params=param[1])

Coordinates: 33.91915512084961°N 73.27930450439453°E
Elevation: 1540.0 m asl
Timezone difference to GMT+0: 18000s

Hourly data
                          date site_code  temperature_2m  relative_humidity_2m  \
0   2026-07-19 19:00:00+00:00     HBRPO       20.557501             96.659309   
1   2026-07-19 20:00:00+00:00     HBRPO       20.307501             96.953217   
2   2026-07-19 21:00:00+00:00     HBRPO       20.057501             96.947495   
3   2026-07-19 22:00:00+00:00     HBRPO       20.307501             96.653069   
4   2026-07-19 23:00:00+00:00     HBRPO       19.907501             97.245934   
..                        ...       ...             ...                   ...   
187 2026-07-27 14:00:00+00:00     HBRPO       22.857500             89.601364   
188 2026-07-27 15:00:00+00:00     HBRPO       22.157501             88.723206   
189 2026-07-27 16:00:00+00:00     HBRPO       21.707500             89.238548   
190 2026-07-27 17:00:00+00:00     HBRPO       21.257502       

In [15]:
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    print(param)

('HBRPO', {'latitude': 33.93991, 'longitude': 73.27844, 'hourly': ['temperature_2m', 'relative_humidity_2m', 'shortwave_radiation'], 'timezone': 'auto', 'start_date': '2026-07-20', 'end_date': '2026-07-27'})
